# Install Requirements

In [1]:
!pip install raiwidgets
!pip install --upgrade pandas

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


  Using cached raiwidgets-0.36.0-py3-none-any.whl.metadata (18 kB)
  Using cached numpy-1.26.2-cp310-cp310-win_amd64.whl.metadata (61 kB)
  Using cached pandas-1.5.3-cp310-cp310-win_amd64.whl.metadata (12 kB)
  Using cached rai_core_flask-0.7.6-py3-none-any.whl.metadata (2.0 kB)
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
  Using cached erroranalysis-0.5.5-py3-none-any.whl.metadata (2.1 kB)
  Using cached fairlearn-0.7.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached raiutils-0.4.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached responsibleai-0.36.0-py3-none-any.whl.metadata (18 kB)
  Using cached dice_ml-0.11-py3-none-any.whl.metadata (20 kB)
  Using cached econml-0.16.0-cp310-cp310-win_amd64.whl.metadata (37 kB)
  Using cached statsmodels-0.13.5-cp310-cp310-win_amd64.whl.metadata (9.5 kB)
  Using cached interpret_community-0.32.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached numba-0.58.1-cp310-cp310-win_amd64.whl.metadata (2.8 kB)
  Using cached sci

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dice-ml 0.11 requires pandas<2.0.0, but you have pandas 2.3.3 which is incompatible.
erroranalysis 0.5.5 requires pandas<2.0.0,>=0.25.1, but you have pandas 2.3.3 which is incompatible.
raiwidgets 0.36.0 requires pandas<2.0.0,>=0.25.1, but you have pandas 2.3.3 which is incompatible.
responsibleai 0.36.0 requires pandas<2.0.0,>=0.25.1, but you have pandas 2.3.3 which is incompatible.


# Do Imports

In [1]:
import os
import json
import mlflow
import pandas as pd
import mlflow.sklearn
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.ml import dsl, Input
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Connect to Azure ML workspace and azureml registry

In [3]:
load_dotenv()

True

In [4]:
subscription_id = os.getenv("SUBSCRIPTION")
resource_group = os.getenv("RESOURCE_GROUP")
workspace_name = os.getenv("WS_NAME")


In [5]:
# Handle to the workspace
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


# Specific RAI Imports (Responsible AI Toolbox)

In [6]:
from raiwidgets import ResponsibleAIDashboard
from responsibleai import RAIInsights

ModuleNotFoundError: No module named 'raiwidgets'

## Define train data, test_data, target_feature

In [ ]:
train_data = Input(type=AssetTypes.MLTABLE, path="./model_training/train_data")
test_data  = Input(type=AssetTypes.MLTABLE, path="./model_training/test_data")
target_feature ="Accept"

# Pull the path to the latest trained model in Azure

In [7]:
import sys
from pathlib import Path
# Add parent directory to path to import command_job.py
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))
from command_job import artifact_path_name


# Get the latest version of the model with the specified name
try:
    models = list(ml_client.models.list(name=artifact_path_name))
    
    if not models:
        print(f"No models found with name '{artifact_path_name}'")
    else:
        # Sort models by creation time (newest first)
        models.sort(key=lambda x: x.creation_context.created_at, reverse=True)
        
        # Get the latest model
        latest_model = models[0]
        
        print(f"Found latest model:")
        print(f"  Name: {latest_model.name}")
        print(f"  Version: {latest_model.version}")
        print(f"  Created: {latest_model.creation_context.created_at}")
        
        # Format the model path for use with the Responsible AI Dashboard
        azureml_model_id = f"azureml:{latest_model.name}:{latest_model.version}"
        print(f"\nModel path for Responsible AI Dashboard:")
        print(f"  {azureml_model_id}")
        
        # Create the model_input object for the RAI dashboard
        model_input = Input(type=AssetTypes.MLFLOW_MODEL, path=azureml_model_id)
        print("\nModel input object created successfully!")
        
except Exception as e:
    print(f"Error retrieving model: {e}")

Found latest model:
  Name: admissions_model
  Version: 12
  Created: 2025-12-23 20:05:03.295514+00:00

Model path for Responsible AI Dashboard:
  azureml:admissions_model:12

Model input object created successfully!


## Define Categorical feature via Feature Metadata

In [8]:
rai_insights = RAIInsights(
        model=model_input,
        train=train_data,
        test=test_data,
        target_column='Admitted',
        task_type='classification',
        categorical_features=[]  # Already encoded, so empty list
    )

c:\Users\micha\miniconda3\envs\admission_env\lib\site-packages\responsibleai\rai_insights\rai_insights.py:338: UserWarning: The categorical_features argument on the RAIInsights constructor is deprecated and will be removed after version 0.26. Please provide categorical features via the feature_metadata argument instead.
  warnings.warn("The categorical_features argument on the "


UserConfigValidationException: Unsupported data type for either train or test. Expecting pandas DataFrame for train and test.

In [27]:
from responsibleai.feature_metadata import FeatureMetadata

feature_metadata = FeatureMetadata(categorical_features=["Gender_Male","Gender_Female", "Race_Asian","Race_Black","Race_Hispanic","Race_White","Race_Other","LegacyStatus","FirstGeneration","FinancialAid"])

In [50]:
rai_insights = RAIInsights(train_data, test_data,  task_type='classification',target_feature,
                             feature_metadata=feature_metadata
                        )

SyntaxError: positional argument follows keyword argument (4058284718.py, line 3)